# Laboratorio 8 — DuckDB
## Ejercicio 3: consultas directas sobre archivos Parquet

Este notebook explora los viajes Yellow Taxi y Green Taxi de 2026 con DuckDB. Las consultas leen los archivos Parquet directamente: no se crea ni se carga una tabla persistente. Las carpetas esperadas son `data/raw/yellow/2026/` y `data/raw/green/2026/`.

**Cobertura del corte analizado:** enero–agosto de 2026, 16 archivos. Los resultados registrados en las celdas Markdown se midieron el 6 de octubre de 2026; cambiarán cuando la TLC publique meses nuevos y se vuelvan a descargar. Ejecute las celdas de código para obtener resultados actualizados.

### Preparación

DuckDB está incluido en el ambiente del laboratorio. El siguiente bloque encuentra la raíz del proyecto tanto si el kernel inicia en `/workspace` como si inicia en `notebooks/`. En VS Code o Jupyter fuera de Docker, ejecute el notebook con el directorio del repositorio como raíz.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

candidatos_raiz = (Path.cwd(), *Path.cwd().parents)
raiz_proyecto = next(
    (ruta for ruta in candidatos_raiz if (ruta / "data" / "raw").is_dir()),
    None,
)
if raiz_proyecto is None:
    raise FileNotFoundError(
        "No se encontro data/raw. Inicie Jupyter desde la raiz del repositorio "
        "o configure el kernel para ejecutarse dentro del proyecto."
    )

yellow_glob = str(raiz_proyecto / "data" / "raw" / "yellow" / "2026" / "*.parquet")
green_glob = str(raiz_proyecto / "data" / "raw" / "green" / "2026" / "*.parquet")
todos_glob = str(raiz_proyecto / "data" / "raw" / "*" / "2026" / "*.parquet")
con = duckdb.connect(database=":memory:")

print(f"Raiz del proyecto: {raiz_proyecto}")
print(f"DuckDB: {duckdb.__version__}")

## 3.1 — Cantidad de archivos disponibles

**Consulta:** contar los nombres de archivo que aparecen en los metadatos Parquet. **Objetivo:** verificar cuántos archivos componen el conjunto. **Fuente:** ambos directorios `data/raw/<tipo>/2026/`. **Resultado observado:** 16 archivos (ocho por tipo). **Decisión:** este corte cubre enero–agosto, no los doce meses del año; la TLC todavía no había publicado septiembre–diciembre.

In [ ]:
consulta_archivos = f"""
SELECT count(DISTINCT file_name) AS archivos
FROM parquet_metadata('{todos_glob}')
"""
display(con.execute(consulta_archivos).df())

## 3.2 — Cantidad de registros y distribución mensual

**Consulta y objetivo:** contar registros por tipo y total, y contrastar los totales con los conteos mensuales. **Fuente:** los Parquet Yellow y Green. El total combinado usa `union_by_name = true` porque ambos esquemas contienen nombres diferentes para sus columnas temporales. **Resultado observado:** 29,703,355 registros Yellow, 337,114 Green y 30,040,469 en total. La suma de los conteos mensuales coincide con el total de cada tipo. **Decisión:** conservar el análisis separado por taxi cuando se usan columnas específicas y unir por nombre solo para operaciones comunes.

In [ ]:
consulta_registros = f"""
SELECT 'yellow' AS tipo, count(*) AS registros
FROM read_parquet('{yellow_glob}')
UNION ALL
SELECT 'green', count(*)
FROM read_parquet('{green_glob}')
UNION ALL
SELECT 'total', count(*)
FROM read_parquet('{todos_glob}', union_by_name = true)
"""
display(con.execute(consulta_registros).df())

consulta_mensual = f"""
SELECT
    CASE WHEN filename LIKE '%yellow%' THEN 'yellow' ELSE 'green' END AS tipo,
    regexp_extract(filename, '2026-([0-9]{{2}})\\.parquet', 1) AS mes,
    count(*) AS registros
FROM read_parquet('{todos_glob}', union_by_name = true, filename = true)
GROUP BY tipo, mes
ORDER BY tipo, mes
"""
registros_mensuales = con.execute(consulta_mensual).df()
display(registros_mensuales.pivot(index="mes", columns="tipo", values="registros"))

## 3.3 y 3.4 — Columnas y tipos de datos

**Consultas:** `DESCRIBE` sobre los archivos de cada taxi por separado. **Objetivo:** obtener los nombres y tipos que DuckDB infiere del esquema Parquet. **Fuente:** los archivos mensuales de cada tipo. **Resultado observado:** 20 columnas Yellow y 21 Green; DuckDB indica que todas pueden contener nulos. Los tipos comunes incluyen `VendorID` (`INTEGER`), fechas (`TIMESTAMP`), `passenger_count` y `RatecodeID` (`BIGINT`), `trip_distance` y montos (`DOUBLE`), y `store_and_fwd_flag` (`VARCHAR`). **Decisión:** mantener los esquemas separados y no renombrar columnas de origen durante la exploración.

In [ ]:
esquema_yellow = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{yellow_glob}')"
).df()
esquema_green = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{green_glob}')"
).df()

print(f"Yellow: {len(esquema_yellow)} columnas")
display(esquema_yellow[["column_name", "column_type", "null"]])
print(f"Green: {len(esquema_green)} columnas")
display(esquema_green[["column_name", "column_type", "null"]])

Los nombres de las columnas temporales distinguen los tipos: Yellow usa `tpep_pickup_datetime` y `tpep_dropoff_datetime`; Green usa `lpep_pickup_datetime` y `lpep_dropoff_datetime`. Yellow contiene `Airport_fee`, mientras Green contiene `ehail_fee` y `trip_type`; Green también incluye `cbd_congestion_fee`. Esto confirma que no se debe asumir que ambos esquemas son idénticos.

## 3.5 — Muestra de registros

**Consulta:** seleccionar tres filas por taxi con `LIMIT 3`. **Objetivo:** inspeccionar registros reales y verificar la legibilidad de campos importantes. **Fuente:** los Parquet Yellow y Green. **Resultado observado:** la primera fila Yellow tiene recogida `2026-01-01 00:54:04`, entrega `00:59:37`, distancia `0.97`, tarifa `7.20` y total `15.86`; la primera fila Green tiene recogida `2026-01-01 00:27:58`, entrega `00:55:16`, distancia `6.20`, tarifa `31.70` y total `45.20`. **Decisión:** usar las columnas `tpep_*` o `lpep_*` correspondientes. La muestra es para inspección, no es aleatoria ni sirve por sí sola para inferencia estadística.

In [ ]:
muestra_yellow = f"""
SELECT VendorID, tpep_pickup_datetime, tpep_dropoff_datetime,
       passenger_count, trip_distance, fare_amount, total_amount
FROM read_parquet('{yellow_glob}')
LIMIT 3
"""
muestra_green = f"""
SELECT VendorID, lpep_pickup_datetime, lpep_dropoff_datetime,
       passenger_count, trip_distance, fare_amount, total_amount
FROM read_parquet('{green_glob}')
LIMIT 3
"""
display(con.execute(muestra_yellow).df())
display(con.execute(muestra_green).df())

## 3.6 — Revisión inicial de calidad

**Consulta:** contar valores nulos y casos potencialmente atípicos: pasajeros en cero, distancia cero o negativa, tarifas y totales negativos, recogidas fuera de 2026, entrega anterior a recogida y duración superior a 24 horas. **Objetivo:** identificar aspectos que investigar antes de limpiar o analizar. **Fuente:** todos los archivos de cada tipo, revisados por separado. **Resultados observados:**

| Indicador | Yellow (29,703,355 filas) | Green (337,114 filas) |
|---|---:|---:|
| Recogida/entrega nula | 0 / 0 | 0 / 0 |
| `passenger_count` nulo | 7,716,688 | 48,775 |
| `passenger_count = 0` | 91,359 | 4,527 |
| `trip_distance = 0` | 952,231 | 12,212 |
| `trip_distance < 0` | 0 | 0 |
| `fare_amount < 0` | 157,364 | 999 |
| `total_amount < 0` | 161,835 | 1,023 |
| Recogida fuera de 2026 | 17 | 14 |
| Entrega anterior a recogida | 10 | 5 |
| Duración mayor a 24 horas | 263 | 4 |

**Decisión:** no se eliminan filas en esta inspección. Nulos y ceros de pasajeros, distancia cero, cargos negativos y duraciones extremas necesitan reglas de limpieza basadas en el contexto. Los cargos negativos podrían representar ajustes o reembolsos. Las recogidas fuera del año también se deben revisar antes de definir el periodo analítico. No se observaron fechas de recogida o entrega nulas ni distancias negativas en este corte.

In [ ]:
def consulta_calidad(glob: str, prefijo_fecha: str, tipo: str) -> str:
    """Construye métricas de calidad usando las columnas temporales del taxi."""
    pickup = f"{prefijo_fecha}_pickup_datetime"
    dropoff = f"{prefijo_fecha}_dropoff_datetime"
    return f"""
    SELECT
        '{tipo}' AS tipo,
        count(*) AS registros,
        countif({pickup} IS NULL) AS pickup_nulo,
        countif({dropoff} IS NULL) AS dropoff_nulo,
        countif(passenger_count IS NULL) AS pasajeros_nulo,
        countif(passenger_count = 0) AS pasajeros_cero,
        countif(trip_distance = 0) AS distancia_cero,
        countif(trip_distance < 0) AS distancia_negativa,
        countif(fare_amount < 0) AS tarifa_negativa,
        countif(total_amount < 0) AS total_negativo,
        countif(year({pickup}) <> 2026) AS pickup_fuera_de_2026,
        countif({dropoff} < {pickup}) AS dropoff_antes_de_pickup,
        countif(date_diff('minute', {pickup}, {dropoff}) > 1440)
            AS duracion_mayor_a_un_dia
    FROM read_parquet('{glob}')
    """

calidad_yellow = con.execute(
    consulta_calidad(yellow_glob, "tpep", "yellow")
).df()
calidad_green = con.execute(
    consulta_calidad(green_glob, "lpep", "green")
).df()
display(pd.concat([calidad_yellow, calidad_green], ignore_index=True).set_index("tipo").T)

## 3.7 y 3.8 — Consultas, fuentes y trazabilidad

Las celdas anteriores incluyen las consultas empleadas para los conteos, esquemas, muestras y métricas de calidad. Cada sección identifica su objetivo, fuente, resultado del corte observado y decisión. Las fuentes son los archivos en `data/raw/yellow/2026/*.parquet` y `data/raw/green/2026/*.parquet`; la consulta combinada lee ambos patrones con `union_by_name = true`. Todas las consultas leen los archivos directamente y no crean tablas persistentes.

## 3.9 — ¿Qué significa consultar Parquet directamente?

DuckDB usa `read_parquet` para leer las columnas y filas directamente de los archivos al ejecutar una consulta, sin importar primero el conjunto completo a una tabla de base de datos. Esta estrategia evita una etapa de importación y una copia adicional, permite pedir solo las columnas necesarias y facilita procesar varios meses mediante patrones de archivos. Parquet es columnar, por lo que las consultas que seleccionan pocas columnas pueden evitar leer las demás. `union_by_name` permite combinar datos por nombre de columna cuando los esquemas difieren o faltan campos.

La lectura directa resulta práctica para explorar volúmenes grandes y añadir nuevos archivos sin rehacer una importación. Si se repiten consultas costosas o se necesitan transformaciones persistentes, puede ser conveniente materializar resultados después; no fue necesario para este ejercicio.